## Part 1

In [ ]:
import requests                         # for sending HTTP requests

from tqdm.notebook import tqdm, trange  # for progress bars

from scrapy import Selector             # for parsing HTML content

## Part 2

### Step 1
Look Wikipedia Controversial topics page

### Step 2
Extract article links

### Step 3
Visit each article talk page

### Step 4
Locate content assessment box

### Step 5
Extract:
 - article name
 - article url
 - content class
 - categories and url

### Step 6
Store data as dictionary

### Step 7
Convert list of dictionaries into pandas DataFrame

## Part 3

In [14]:
article_url = 'https://en.wikipedia.org/wiki/Meaning_of_life'

In [15]:
headers = {
    'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

In [16]:
response = requests.get(article_url, headers=headers)
sel = Selector(text=response.text)

In [17]:
name = sel.css('#firstHeading span::text').get()
print(name)

Meaning of life


In [19]:
talk_url = article_url.replace('/wiki/', '/wiki/Talk:')
response_talk = requests.get(talk_url, headers=headers)
sel_talk = Selector(text=response_talk.text)

In [20]:
texts= sel_talk.css('::text').getall()

content_class = None
for t in texts:
    t = t.strip()
    if "class" in t.lower():
        if "b-class" in t.lower():
            content_class = "B-class"
        elif "c-class" in t.lower():
            content_class = "C-class"
        elif "start-class" in t.lower():
            content_class = "Start-class"
        elif "stub-class" in t.lower():
            content_class = "Stub-class"
print(content_class)

B-class


In [21]:
categories = {}

base_url = 'https://en.wikipedia.org'

for a in sel_talk.css("a"):
    href = a.attrib.get('href', '')
    text = a.css('::text').get()

    if href and "Category:"  in href:
        categories[text] = base_url + href

print(categories)

{'High': 'https://en.wikipedia.org/wiki/Category:High-importance_Religion_articles', 'High-importance': 'https://en.wikipedia.org/wiki/Category:High-importance_Latter_Day_Saint_movement_articles', 'Mid-importance': 'https://en.wikipedia.org/wiki/Category:Mid-importance_Anglicanism_articles', 'Wikipedia controversial topics': 'https://en.wikipedia.org/wiki/Category:Wikipedia_controversial_topics', 'Old requests for peer review': 'https://en.wikipedia.org/wiki/Category:Old_requests_for_peer_review', 'B-Class level-5 vital articles': 'https://en.wikipedia.org/wiki/Category:B-Class_level-5_vital_articles', 'Wikipedia level-5 vital articles in Philosophy and religion': 'https://en.wikipedia.org/wiki/Category:Wikipedia_level-5_vital_articles_in_Philosophy_and_religion', 'B-Class vital articles in Philosophy and religion': 'https://en.wikipedia.org/wiki/Category:B-Class_vital_articles_in_Philosophy_and_religion', 'B-Class Spirituality articles': 'https://en.wikipedia.org/wiki/Category:B-Class

In [22]:
import pandas as pd

In [23]:
data = []

In [24]:
data ={
    'name': name,
    'url': article_url,
    'content_assessment_class': content_class,
    'categories ': categories
}

In [25]:
df = pd.DataFrame([data])

In [26]:
df

,name,url,content_assessment_class,categories
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,{'High': 'https://en.wikipedia.org/wiki/Catego...


## Method 1 - json_normalize() (Dictionary to columns)

In [27]:
cat_df = pd.json_normalize(df['categories '])
df_new = pd.concat([df.drop('categories ', axis=1), cat_df], axis=1)    
df_new

,name,url,content_assessment_class,High,High-importance,Mid-importance,Wikipedia controversial topics,Old requests for peer review,B-Class level-5 vital articles,Wikipedia level-5 vital articles in Philosophy and religion,...,Unknown-importance Reformed Christianity articles,WikiProject Reformed Christianity articles,B-Class Latter Day Saint movement articles,High-importance Latter Day Saint movement articles,WikiProject Latter Day Saint movement articles,WikiProject Christianity articles,B-Class Religion articles,High-importance Religion articles,WikiProject Religion articles,All Wikipedia vital articles
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,https://en.wikipedia.org/wiki/Category:High-im...,https://en.wikipedia.org/wiki/Category:High-im...,https://en.wikipedia.org/wiki/Category:Mid-imp...,https://en.wikipedia.org/wiki/Category:Wikiped...,https://en.wikipedia.org/wiki/Category:Old_req...,https://en.wikipedia.org/wiki/Category:B-Class...,https://en.wikipedia.org/wiki/Category:Wikiped...,...,https://en.wikipedia.org/wiki/Category:Unknown...,https://en.wikipedia.org/wiki/Category:WikiPro...,https://en.wikipedia.org/wiki/Category:B-Class...,https://en.wikipedia.org/wiki/Category:High-im...,https://en.wikipedia.org/wiki/Category:WikiPro...,https://en.wikipedia.org/wiki/Category:WikiPro...,https://en.wikipedia.org/wiki/Category:B-Class...,https://en.wikipedia.org/wiki/Category:High-im...,https://en.wikipedia.org/wiki/Category:WikiPro...,https://en.wikipedia.org/wiki/Category:All_Wik...


## Method 2 - explode() (Dictionary to Multiple rows)

In [29]:
df['categories'] = df['categories '].apply(lambda x: list(x.items()))

In [30]:
df_exploded = df.explode('categories')

In [31]:
df_exploded[['category_name', 'category_url']] = pd.DataFrame(df_exploded['categories'].tolist(), index=df_exploded.index)

In [32]:
df_exploded.drop(columns = ['categories '], inplace=True)

In [33]:
df_exploded

,name,url,content_assessment_class,categories,category_name,category_url
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(High, https://en.wikipedia.org/wiki/Category:...",High,https://en.wikipedia.org/wiki/Category:High-im...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(High-importance, https://en.wikipedia.org/wik...",High-importance,https://en.wikipedia.org/wiki/Category:High-im...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(Mid-importance, https://en.wikipedia.org/wiki...",Mid-importance,https://en.wikipedia.org/wiki/Category:Mid-imp...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(Wikipedia controversial topics, https://en.wi...",Wikipedia controversial topics,https://en.wikipedia.org/wiki/Category:Wikiped...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(Old requests for peer review, https://en.wiki...",Old requests for peer review,https://en.wikipedia.org/wiki/Category:Old_req...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(B-Class level-5 vital articles, https://en.wi...",B-Class level-5 vital articles,https://en.wikipedia.org/wiki/Category:B-Class...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,(Wikipedia level-5 vital articles in Philosoph...,Wikipedia level-5 vital articles in Philosophy...,https://en.wikipedia.org/wiki/Category:Wikiped...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,(B-Class vital articles in Philosophy and reli...,B-Class vital articles in Philosophy and religion,https://en.wikipedia.org/wiki/Category:B-Class...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(B-Class Spirituality articles, https://en.wik...",B-Class Spirituality articles,https://en.wikipedia.org/wiki/Category:B-Class...
0,Meaning of life,https://en.wikipedia.org/wiki/Meaning_of_life,B-class,"(High-importance Spirituality articles, https:...",High-importance Spirituality articles,https://en.wikipedia.org/wiki/Category:High-im...
